# Silver - TU PortPS tulip - unit tests + data-quality validation

Tulip variant of `TU_PortPS_silver_tests.ipynb` (kept separately). Adds the `request_id` unit
tests and asserts `request_id` uniqueness / non-null in the DQ validation. Reads the tulip table
`tu_portps_tulip`.

Two parts:

1. **Unit / invariant tests** - exercise the cleaning + feature logic on tiny synthetic fixtures (no S3). The functions here are copied/adapted from `TU_PortPS_tulip_silver.ipynb`; keep them in sync.
2. **DQ validation** - read the written silver table and run completeness / uniqueness / consistency / timeliness / referential-integrity checks, then save results to `dq/{results,checks,issues}/`.

In [1]:
%%pyspark project.spark.compatibility
from pyspark.sql import SparkSession, DataFrame, Row
from pyspark.sql import functions as F
from pyspark.sql.functions import lit, col, udf
from pyspark.sql.window import Window
from pyspark.sql.types import (
    StructType, StructField, StringType, IntegerType, LongType, BooleanType,
    TimestampType, DateType, ArrayType, MapType,
)
import xml.etree.ElementTree as ET
import contextlib

spark = SparkSession.builder.appName("tu_portps_tulip_silver_tests").getOrCreate()

Executing for connection type: SPARK_GLUE, connection name: project.spark.compatibility
Creating Glue session...
Create session for connection: project.spark.compatibility


'Session ae2xwmv6edkt56-049a7dd6-4888-4db4-bfe2-f463db5fa369 has been created.'

<sagemaker_studio_dataengineering_sessions.sagemaker_base_session_manager.common.debugging_utils.SessionInfoTableDisplay object>

Session created for connection: project.spark.compatibility.

Connection: project.spark.compatibility | Run start time: 2026-07-30 20:43:42.782907 | Run duration : 0:02:15.764624s.


## Cleaning + feature logic (copied from TU_PortPS_tulip_silver.ipynb - keep in sync)

In [2]:
%%pyspark project.spark.compatibility
# XML parse (mirrors the cleaning notebook).
parse_struct = StructType([
    StructField("parse_ok",     BooleanType(),                                  True),
    StructField("phone_number", StringType(),                                   True),
    StructField("snapshot",     MapType(StringType(), StringType()),            True),
    StructField("events",       ArrayType(MapType(StringType(), StringType())), True),
])

def _local(tag):
    return tag.split("}", 1)[1] if "}" in tag else tag

def parse_tnresponse(xml_str):
    if not xml_str or not xml_str.strip():
        return (False, None, {}, [])
    try:
        root = ET.fromstring(xml_str)
        if "}" in root.tag:
            return (False, None, {}, [])
        snapshot = {}
        def walk(el, prefix):
            for ch in el:
                name = _local(ch.tag)
                if name == "PortingHistory":
                    continue
                path = (prefix + "/" + name) if prefix else name
                if len(ch):
                    walk(ch, path)
                else:
                    text = ch.text.strip() if (ch.text and ch.text.strip()) else None
                    if text is not None:
                        snapshot[path] = text
        walk(root, "")
        events = []
        ph = root.find("PortingHistory")
        if ph is not None:
            for pi in ph.findall("PortingInfo"):
                m = {}
                for ch in pi:
                    text = ch.text.strip() if (ch.text and ch.text.strip()) else None
                    if text is not None:
                        m[_local(ch.tag)] = text
                events.append(m)
        return (True, snapshot.get("Ownership/PhoneNumber"), snapshot, events)
    except Exception:
        return (False, None, {}, [])


Connection: project.spark.compatibility | Run start time: 2026-07-30 20:45:58.552552 | Run duration : 0:00:00.367697s.


In [3]:
%%pyspark project.spark.compatibility
# norm + vocabularies (mirror the cleaning notebook).
def norm(c):
    c = F.trim(c)
    return F.when(c.isin("", "NULL", "\\N"), None).otherwise(c)

EVENT_TYPE_MAP = {
    "Inter-SPID Port":              "INTER_SPID_PORT",
    "Intra-SPID Port":              "INTRA_SPID_PORT",
    "NPAC Code Assignment":         "NPAC_CODE_ASSIGNMENT",
    "Return to NPAC Code Assignee": "RETURN_TO_NPAC_CODE_ASSIGNEE",
    "NPAC SPID Update":             "NPAC_SPID_UPDATE",
    "NPAC SPID Migration":          "NPAC_SPID_MIGRATION",
}
LNP_TYPE_MAP = {"INTER_SPID_PORT": 0, "INTRA_SPID_PORT": 1}
# sv_type allowed values (TU API guide, Appendix B "SV Type"). Unknown values go to dq/issues for review.
SV_TYPE_VOCAB = ["Wireline", "Wireless", "VoWIFI", "VoIP", "Pre-Paid Wireless",
                 "Class 2 Interconnected VoIP"]

# SPID dict, trimmed to what the tests touch (full maps in the cleaning notebook).
SPID_TO_CARRIER = {
    "6574": "BELL_MOBILITY", "5643": "FIDO", "8820": "FIDO", "497E": "SHAW",
    "8084": "TELUS", "8303": "TELUS_MOBILITY", "328F": "VIDEOTRON", "8306": "VIDEOTRON",
    "383F": "FREEDOM", "9868": "SASKTEL", "8088": "MTS", "329A": "ALIANT",
}
CARRIER_PARENT = {
    "BELL": "BELL", "BELL_MOBILITY": "BELL", "ALIANT": "BELL", "MTS": "BELL",
    "ROGERS": "ROGERS", "FIDO": "ROGERS", "SHAW": "ROGERS",
    "TELUS": "TELUS", "TELUS_MOBILITY": "TELUS",
}
BIG3 = ["BELL", "ROGERS", "TELUS"]


Connection: project.spark.compatibility | Run start time: 2026-07-30 20:45:58.924016 | Run duration : 0:00:00.873788s.


In [4]:
%%pyspark project.spark.compatibility
# Feature logic as functions for unit tests. Mirror the cleaning notebook.

def add_event_type(df):
    et_map = F.create_map([x for k, v in EVENT_TYPE_MAP.items() for x in (lit(k), lit(v))])
    df = df.withColumn("event_type", et_map[norm(F.col("raw_event"))])
    df = df.withColumn("_event_type_unknown",
                       norm(F.col("raw_event")).isNotNull() & F.col("event_type").isNull())
    return df

def add_lnp_type(df):
    lnp_map = F.create_map([x for k, v in LNP_TYPE_MAP.items() for x in (lit(k), lit(v))])
    df = df.withColumn("lnp_type", lnp_map[F.col("event_type")])
    df = df.withColumn("is_inter_carrier_port", F.coalesce(F.col("lnp_type") == lit(0), lit(False)))
    return df

def add_record_id(df):
    # record_id = 64-hex sha256 of the natural key.
    return df.withColumn("record_id", F.sha2(F.concat_ws("|",
        F.col("phone_number_AC_hash"),
        F.col("event_timestamp").cast("string"),
        F.coalesce(F.col("to_spid"), lit("")),
        F.coalesce(F.col("event_type"), lit("")),
    ), 256))

def add_event_seq_and_from_spid(df):
    w = Window.partitionBy("_phone").orderBy(F.col("event_timestamp").asc())
    df = df.withColumn("event_seq", F.row_number().over(w))
    df = df.withColumn("_prev_to_spid",    F.lag("to_spid").over(w))
    df = df.withColumn("_prev_event_type", F.lag("event_type").over(w))
    df = df.withColumn("from_spid",
        F.when(F.col("_prev_to_spid").isNull(), None)                                  # first event
         .when(F.col("event_type") == lit("NPAC_CODE_ASSIGNMENT"), None)
         .when(F.col("_prev_event_type") == lit("RETURN_TO_NPAC_CODE_ASSIGNEE"), None)  # recycle boundary
         .otherwise(F.col("_prev_to_spid")))
    return df

def add_mno(df, spid_col, raw_out, mno_out):
    brand_map  = F.create_map([x for k, v in SPID_TO_CARRIER.items() for x in (lit(k), lit(v))])
    parent_map = F.create_map([x for k, v in CARRIER_PARENT.items()  for x in (lit(k), lit(v))])
    big3_arr   = F.array(*[lit(c) for c in BIG3])
    df = df.withColumn(raw_out, brand_map[F.col(spid_col)])
    df = df.withColumn("_parent_tmp", parent_map[F.col(raw_out)])
    df = df.withColumn(mno_out,
                       F.when(F.array_contains(big3_arr, F.col("_parent_tmp")), F.col("_parent_tmp")).otherwise(None))
    return df.drop("_parent_tmp")

def add_from_to_same_carrier(df):
    # 1 = same raw brand, 0 = different, null when either side is null.
    return df.withColumn("from_to_same_carrier",
        F.when(F.col("from_mno_raw").isNull() | F.col("to_mno_raw").isNull(), None)
         .otherwise((F.col("from_mno_raw") == F.col("to_mno_raw")).cast("int")))


def add_request_id(df):
    # request_id = deterministic UUID from bronze id + natural key (unique, non-null). Mirrors
    # the tulip cleaning notebook; production reads it from the bronze request_id column.
    seed = F.concat_ws("|",
        F.col("bronze_row_id").cast("string"),
        F.col("phone_number_AC_hash"),
        F.col("event_timestamp").cast("string"),
        F.coalesce(F.col("to_spid"), lit("")),
        F.coalesce(F.col("event_type"), lit("")),
    )
    h = F.sha2(seed, 256)
    return df.withColumn("request_id", F.concat_ws("-",
        F.substring(h, 1, 8),  F.substring(h, 9, 4),  F.substring(h, 13, 4),
        F.substring(h, 17, 4), F.substring(h, 21, 12)))



Connection: project.spark.compatibility | Run start time: 2026-07-30 20:45:59.802078 | Run duration : 0:00:00.266692s.


## Test fixtures

In [5]:
%%pyspark project.spark.compatibility
# Synthetic <TNResponse> docs for the parser variants.
OK_XML = (
    "<TNResponse>"
    "  <Ownership><PhoneNumber>4165551234</PhoneNumber><Spid>6574</Spid>"
    "    <Company>BELL MOBILITY</Company></Ownership>"
    "  <CodeInfo><CnaCodeOwner>BELL</CnaCodeOwner><Ocn>6574</Ocn></CodeInfo>"
    "  <PortingHistory>"
    "    <PortingInfo><Date>2020-01-15 00:00:00</Date><Event>Inter-SPID Port</Event>"
    "      <Spid>6574</Spid><LRN>416-555-0000</LRN><SvType>wireline</SvType></PortingInfo>"
    "  </PortingHistory>"
    "</TNResponse>"
)

# Valid 3000, no porting history (e.g. a US number).
EMPTY_XML = (
    "<TNResponse>"
    "  <Ownership><PhoneNumber>2025551234</PhoneNumber>"
    "    <Company>No customer agreement for region: Northeast</Company></Ownership>"
    "</TNResponse>"
)

# Two events in one response (history is new->old in the source).
MULTI_XML = (
    "<TNResponse>"
    "  <Ownership><PhoneNumber>5145550000</PhoneNumber><Spid>328F</Spid></Ownership>"
    "  <PortingHistory>"
    "    <PortingInfo><Date>2021-06-01 00:00:00</Date><Event>Inter-SPID Port</Event><Spid>328F</Spid></PortingInfo>"
    "    <PortingInfo><Date>2018-01-01 00:00:00</Date><Event>NPAC Code Assignment</Event><Spid>6574</Spid></PortingInfo>"
    "  </PortingHistory>"
    "</TNResponse>"
)

# Namespaced root -> parser marks it bad.
NS_XML = '<TNResponse xmlns="http://example.com/tn"><Ownership><Spid>6574</Spid></Ownership></TNResponse>'


Connection: project.spark.compatibility | Run start time: 2026-07-30 20:46:00.073151 | Run duration : 0:00:00.859274s.


## Unit / invariant tests

In [6]:
%%pyspark project.spark.compatibility
def _df(rows):
    """DataFrame from a list of Rows."""
    return spark.createDataFrame(rows)

def _ts(df):
    """Cast event_timestamp string to timestamp."""
    return df.withColumn("event_timestamp", F.col("event_timestamp").cast("timestamp"))

@contextlib.contextmanager
def _toronto_session():
    """Pin session tz to America/Toronto for a test."""
    prev = spark.conf.get("spark.sql.session.timeZone")
    spark.conf.set("spark.sql.session.timeZone", "America/Toronto")
    try:
        yield
    finally:
        spark.conf.set("spark.sql.session.timeZone", prev)


Connection: project.spark.compatibility | Run start time: 2026-07-30 20:46:00.936929 | Run duration : 0:00:00.797346s.


In [7]:
%%pyspark project.spark.compatibility
# XML parse (pure Python).
def test_parse_ok_single_event():
    ok, phone, snap, events = parse_tnresponse(OK_XML)
    assert ok is True
    assert phone == "4165551234"
    assert snap["Ownership/Spid"] == "6574"
    assert snap["CodeInfo/CnaCodeOwner"] == "BELL"
    assert len(events) == 1
    assert events[0]["Event"] == "Inter-SPID Port" and events[0]["Spid"] == "6574"

def test_parse_no_events_response():
    ok, phone, snap, events = parse_tnresponse(EMPTY_XML)
    assert ok is True
    assert events == []
    assert snap["Ownership/Company"].startswith("No customer agreement")

def test_parse_multiple_events_in_document_order():
    ok, phone, snap, events = parse_tnresponse(MULTI_XML)
    assert ok is True and len(events) == 2
    assert events[0]["Spid"] == "328F" and events[1]["Spid"] == "6574"

def test_parse_snapshot_excludes_history_leaves():
    _, _, snap, _ = parse_tnresponse(OK_XML)
    assert all("PortingHistory" not in k for k in snap.keys())

def test_parse_namespace_marked_bad():
    ok, _, _, events = parse_tnresponse(NS_XML)
    assert ok is False and events == []

def test_parse_bad_and_empty_inputs():
    assert parse_tnresponse("not xml <<<")[0] is False
    assert parse_tnresponse("")[0] is False
    assert parse_tnresponse(None)[0] is False
    assert parse_tnresponse("   ")[0] is False


Connection: project.spark.compatibility | Run start time: 2026-07-30 20:46:01.738157 | Run duration : 0:00:00.279214s.


In [8]:
%%pyspark project.spark.compatibility
# norm + vocabularies.
def test_norm_nulls_empty_and_trim():
    rows = [Row(v=""), Row(v="NULL"), Row(v="\\N"), Row(v="  x  "), Row(v="y"), Row(v="   ")]
    out = [r[0] for r in _df(rows).select(norm(F.col("v"))).collect()]
    assert out == [None, None, None, "x", "y", None]

def test_event_type_known_and_unknown():
    rows = [Row(raw_event="Inter-SPID Port"), Row(raw_event="NPAC SPID Update"),
            Row(raw_event="Totally Unknown"), Row(raw_event=None)]
    out = add_event_type(_df(rows)).select("event_type", "_event_type_unknown").collect()
    assert out[0]["event_type"] == "INTER_SPID_PORT" and out[0]["_event_type_unknown"] is False
    assert out[1]["event_type"] == "NPAC_SPID_UPDATE"
    assert out[2]["event_type"] is None and out[2]["_event_type_unknown"] is True   # unknown is flagged
    assert out[3]["event_type"] is None and out[3]["_event_type_unknown"] is False  # null in -> not flagged

def test_lnp_type_and_inter_flag():
    rows = [Row(event_type="INTER_SPID_PORT"), Row(event_type="INTRA_SPID_PORT"),
            Row(event_type="NPAC_CODE_ASSIGNMENT")]
    out = add_lnp_type(_df(rows)).select("lnp_type", "is_inter_carrier_port").collect()
    assert out[0]["lnp_type"] == 0 and out[0]["is_inter_carrier_port"] is True
    assert out[1]["lnp_type"] == 1 and out[1]["is_inter_carrier_port"] is False
    assert out[2]["lnp_type"] is None and out[2]["is_inter_carrier_port"] is False


Connection: project.spark.compatibility | Run start time: 2026-07-30 20:46:02.021139 | Run duration : 0:00:00.726538s.


In [9]:
%%pyspark project.spark.compatibility
# SPID -> carrier.
def test_mno_brand_and_parent_rollup():
    rows = [Row(to_spid="5643"), Row(to_spid="6574"), Row(to_spid="328F"),
            Row(to_spid="9999"), Row(to_spid=None)]
    out = add_mno(_df(rows), "to_spid", "to_mno_raw", "to_mno_std").select("to_spid", "to_mno_raw", "to_mno_std").collect()
    by = {r["to_spid"]: (r["to_mno_raw"], r["to_mno_std"]) for r in out}
    assert by["5643"] == ("FIDO", "ROGERS")          # flanker -> big-3 parent
    assert by["6574"] == ("BELL_MOBILITY", "BELL")
    assert by["328F"] == ("VIDEOTRON", None)          # known brand, not big-3 -> std null
    assert by["9999"] == (None, None)                 # unknown spid
    assert by[None]  == (None, None)


Connection: project.spark.compatibility | Run start time: 2026-07-30 20:46:02.751311 | Run duration : 0:00:00.265375s.


In [10]:
%%pyspark project.spark.compatibility
# record_id.
def test_record_id_deterministic_and_uses_to_spid():
    base = dict(phone_number_AC_hash="h1", event_timestamp="2020-01-01 00:00:00", event_type="INTER_SPID_PORT")
    rows = [Row(**base, to_spid="A"), Row(**base, to_spid="A"), Row(**base, to_spid="B")]
    df = add_record_id(_ts(_df(rows)))
    ids = [r[0] for r in df.select("record_id").collect()]
    assert ids[0] == ids[1]                # same inputs -> same id (deterministic / replay-safe)
    assert ids[0] != ids[2]                # to_spid is part of the 4-column key
    assert all(len(x) == 64 for x in ids)  # full sha256 hex

def test_from_to_same_carrier():
    rows = [Row(from_mno_raw="FIDO", to_mno_raw="FIDO"),   # same brand -> 1
            Row(from_mno_raw="BELL", to_mno_raw="TELUS"),  # different  -> 0
            Row(from_mno_raw=None,   to_mno_raw="TELUS")]  # one side null -> null
    vals = [r[0] for r in add_from_to_same_carrier(_df(rows)).select("from_to_same_carrier").collect()]
    assert vals == [1, 0, None]


Connection: project.spark.compatibility | Run start time: 2026-07-30 20:46:03.020334 | Run duration : 0:00:00.752797s.


In [11]:
%%pyspark project.spark.compatibility
# event_seq + from_spid.
def test_event_seq_orders_by_time_not_input_order():
    rows = [
        Row(_phone="p", event_timestamp="2020-03-01 00:00:00", to_spid="C", event_type="INTER_SPID_PORT"),
        Row(_phone="p", event_timestamp="2020-01-01 00:00:00", to_spid="A", event_type="INTER_SPID_PORT"),
        Row(_phone="p", event_timestamp="2020-02-01 00:00:00", to_spid="B", event_type="INTER_SPID_PORT"),
    ]
    out = add_event_seq_and_from_spid(_ts(_df(rows))).select("to_spid", "event_seq").collect()
    seq = {r["to_spid"]: r["event_seq"] for r in out}
    assert seq["A"] == 1 and seq["B"] == 2 and seq["C"] == 3   # by time, not input order

def test_from_spid_three_null_rules():
    rows = [
        Row(_phone="p", event_timestamp="2020-01-01 00:00:00", to_spid="A", event_type="INTER_SPID_PORT"),
        Row(_phone="p", event_timestamp="2020-02-01 00:00:00", to_spid="B", event_type="INTER_SPID_PORT"),
        Row(_phone="p", event_timestamp="2020-03-01 00:00:00", to_spid="C", event_type="NPAC_CODE_ASSIGNMENT"),
        Row(_phone="p", event_timestamp="2020-04-01 00:00:00", to_spid="D", event_type="RETURN_TO_NPAC_CODE_ASSIGNEE"),
        Row(_phone="p", event_timestamp="2020-05-01 00:00:00", to_spid="E", event_type="INTER_SPID_PORT"),
    ]
    out = add_event_seq_and_from_spid(_ts(_df(rows))).select("to_spid", "from_spid").collect()
    frm = {r["to_spid"]: r["from_spid"] for r in out}
    assert frm["A"] is None      # first event
    assert frm["B"] == "A"       # normal lag
    assert frm["C"] is None      # this row is NPAC_CODE_ASSIGNMENT
    assert frm["D"] == "C"       # RETURN row itself keeps its lagged from
    assert frm["E"] is None      # previous row was RETURN -> recycle boundary


Connection: project.spark.compatibility | Run start time: 2026-07-30 20:46:03.776994 | Run duration : 0:00:00.248099s.


## request_id (tulip)

In [12]:
%%pyspark project.spark.compatibility
# request_id (tulip): deterministic UUID, unique + non-null per event.
import re
_UUID_RE = r"^[0-9a-f]{8}-[0-9a-f]{4}-[0-9a-f]{4}-[0-9a-f]{4}-[0-9a-f]{12}$"

def test_request_id_unique_nonnull_uuid_shape():
    rows = [
        Row(bronze_row_id=1, phone_number_AC_hash="h1", event_timestamp="2020-01-01 00:00:00", to_spid="A", event_type="INTER_SPID_PORT"),
        Row(bronze_row_id=1, phone_number_AC_hash="h1", event_timestamp="2020-01-01 00:00:00", to_spid="B", event_type="INTER_SPID_PORT"),
        Row(bronze_row_id=2, phone_number_AC_hash="h1", event_timestamp="2020-01-01 00:00:00", to_spid="A", event_type="INTER_SPID_PORT"),
    ]
    ids = [r[0] for r in add_request_id(_ts(_df(rows))).select("request_id").collect()]
    assert all(x is not None for x in ids)                       # non-null
    assert len(set(ids)) == len(ids)                             # unique per row (to_spid / bronze_row_id vary)
    assert all(re.match(_UUID_RE, x) for x in ids)               # UUID shape

def test_request_id_deterministic():
    row = [Row(bronze_row_id=7, phone_number_AC_hash="h9", event_timestamp="2021-05-05 00:00:00", to_spid="Z", event_type="INTRA_SPID_PORT")]
    a = add_request_id(_ts(_df(row))).select("request_id").collect()[0][0]
    b = add_request_id(_ts(_df(row))).select("request_id").collect()[0][0]
    assert a == b                                                # replay-stable


Connection: project.spark.compatibility | Run start time: 2026-07-30 20:46:04.029275 | Run duration : 0:00:00.770682s.


In [13]:
%%pyspark project.spark.compatibility
# Run all unit tests
_tests = [
    test_parse_ok_single_event,
    test_parse_no_events_response,
    test_parse_multiple_events_in_document_order,
    test_parse_snapshot_excludes_history_leaves,
    test_parse_namespace_marked_bad,
    test_parse_bad_and_empty_inputs,
    test_norm_nulls_empty_and_trim,
    test_event_type_known_and_unknown,
    test_lnp_type_and_inter_flag,
    test_mno_brand_and_parent_rollup,
    test_record_id_deterministic_and_uses_to_spid,
    test_from_to_same_carrier,
    test_event_seq_orders_by_time_not_input_order,
    test_from_spid_three_null_rules,
    test_request_id_unique_nonnull_uuid_shape,
    test_request_id_deterministic,
]
_failed = 0
for _t in _tests:
    try:
        _t(); print("PASS  " + _t.__name__)
    except AssertionError as e:
        _failed += 1; print("FAIL  " + _t.__name__ + ": " + str(e))
    except Exception as e:
        _failed += 1; print("ERROR " + _t.__name__ + ": " + type(e).__name__ + ": " + str(e))
print()
print(str(len(_tests) - _failed) + "/" + str(len(_tests)) + " passed")

PASS  test_parse_ok_single_event
PASS  test_parse_no_events_response
PASS  test_parse_multiple_events_in_document_order
PASS  test_parse_snapshot_excludes_history_leaves
PASS  test_parse_namespace_marked_bad
PASS  test_parse_bad_and_empty_inputs
PASS  test_norm_nulls_empty_and_trim
PASS  test_event_type_known_and_unknown
PASS  test_lnp_type_and_inter_flag
PASS  test_mno_brand_and_parent_rollup
PASS  test_record_id_deterministic_and_uses_to_spid
PASS  test_from_to_same_carrier
PASS  test_event_seq_orders_by_time_not_input_order
PASS  test_from_spid_three_null_rules
PASS  test_request_id_unique_nonnull_uuid_shape
PASS  test_request_id_deterministic

16/16 passed

Connection: project.spark.compatibility | Run start time: 2026-07-30 20:46:04.803879 | Run duration : 0:00:54.804148s.


## Data-quality validation (reads the written silver table)

Run after `TU_PortPS_tulip_silver.ipynb` has written the table. Saves to `s3://enstream-lake-silver-dev/poc/dq/{results,checks,issues}/dataset=tu_portps_tulip/`. The write lines are commented out - uncomment to persist.

In [14]:
%%pyspark project.spark.compatibility
from datetime import datetime
from zoneinfo import ZoneInfo

SILVER_TABLE = "spark_catalog.trust_score_v1_poc_silver.tu_portps_tulip"
ACCT_CHANGES_TABLE = "spark_catalog.trust_score_v1_poc_silver.account_changes_batch"

BRONZE_PATH          = "s3://enstream-lake-bronze-dev/landing/BNC_processed.csv"
BRONZE_JOIN_KEY_PATH = "s3://enstream-lake-bronze-dev/landing/BNC_hashed.csv"
OK_CODE = "3000"

DQ_BASE    = "s3://enstream-lake-silver-dev/poc/dq/"
DQ_CHECKS  = DQ_BASE + "checks/dataset=tu_portps_tulip/"
DQ_ISSUES  = DQ_BASE + "issues/dataset=tu_portps_tulip/"
DQ_RESULTS = DQ_BASE + "results/dataset=tu_portps_tulip/"

RUN_TS = datetime.now(ZoneInfo("America/Toronto")).replace(tzinfo=None, microsecond=0)

silver = spark.table(SILVER_TABLE)
total = silver.count()
print("silver rows:", total)

silver rows: 8894

Connection: project.spark.compatibility | Run start time: 2026-07-30 20:46:59.612560 | Run duration : 0:00:05.379037s.


In [15]:
%%pyspark project.spark.compatibility
required = silver.columns
rows = []

# completeness: MUST_NONNULL held to 1.0; others record the ratio but always pass.
MUST_NONNULL = ("request_id", "record_id", "phone_number_AC_hash", "phone_number_AT_hash",
                "event_timestamp", "event_date", "event_type", "event_seq",
                "is_inter_carrier_port", "to_spid",
                "source_name", "schema_version", "ingestion_ts", "last_updated_ts")

for c in required:
    nn = silver.filter(F.col(c).isNotNull()).count()
    ratio = (nn / total if total else None)
    must = c in MUST_NONNULL
    threshold = 1.0 if must else 0.0
    passed = (nn == total) if must else True
    rows.append(("completeness", "non_null_" + c, ratio, threshold, passed))


Connection: project.spark.compatibility | Run start time: 2026-07-30 20:47:04.997344 | Run duration : 0:02:26.254160s.


In [16]:
%%pyspark project.spark.compatibility
# uniqueness: request_id (tulip key), record_id, and its 4-column natural-key preimage.
distinct_req = silver.select("request_id").distinct().count()
rows.append(("uniqueness", "distinct_request_id_ratio",
             (distinct_req / total if total else None), 1.0, distinct_req == total))
distinct_pk = silver.select("record_id").distinct().count()
rows.append(("uniqueness", "distinct_record_id_ratio",
             (distinct_pk / total if total else None), 1.0, distinct_pk == total))
nat = silver.select("phone_number_AC_hash", "event_timestamp", "to_spid", "event_type").distinct().count()
rows.append(("uniqueness", "distinct_natural_key_ratio",
             (nat / total if total else None), 1.0, nat == total))



Connection: project.spark.compatibility | Run start time: 2026-07-30 20:49:31.652685 | Run duration : 0:00:21.136989s.


In [17]:
%%pyspark project.spark.compatibility
# consistency
# 1) is_inter_carrier_port <=> lnp_type==0.
inter_inconsistent = silver.filter(
    (F.col("is_inter_carrier_port") & ((F.col("lnp_type") != 0) | F.col("lnp_type").isNull())) |
    (~F.col("is_inter_carrier_port") & (F.col("lnp_type") == 0))
).count()
rows.append(("consistency", "is_inter_carrier_port_matches_lnp_type", float(inter_inconsistent), 0.0, inter_inconsistent == 0))

# 2) event_date == date(event_timestamp).
date_mismatch = silver.filter(F.col("event_date") != F.to_date("event_timestamp")).count()
rows.append(("consistency", "event_date_eq_timestamp_date", float(date_mismatch), 0.0, date_mismatch == 0))

# 3) event_seq contiguous 1..n per number.
seq_chk = silver.groupBy("phone_number_AC_hash").agg(
    F.count("*").alias("n"), F.max("event_seq").alias("mx"), F.countDistinct("event_seq").alias("dc"))
seq_broken = seq_chk.filter((F.col("n") != F.col("mx")) | (F.col("n") != F.col("dc"))).count()
rows.append(("consistency", "event_seq_contiguous_per_phone", float(seq_broken), 0.0, seq_broken == 0))

# validity: values in the allowed domain (event_type, lnp_type, sv_type, lrn, SPID length).
bad_event_type = ~F.col("event_type").isin("INTER_SPID_PORT", "INTRA_SPID_PORT")
bad_lnp_type   = ~F.col("lnp_type").isin(0, 1)
bad_sv_type    = F.col("sv_type").isNotNull() & ~F.col("sv_type").isin(*SV_TYPE_VOCAB)
bad_lrn        = F.col("lrn").isNotNull() & ~F.col("lrn").rlike("^[0-9]{10}$")
bad_spid_len   = (F.length("to_spid") != 4) | (F.col("from_spid").isNotNull() & (F.length("from_spid") != 4))

invalid_validity = silver.filter(
    bad_event_type | bad_lnp_type | bad_sv_type | bad_lrn | bad_spid_len
).count()
rows.append(("validity", "values_in_allowed_domain", float(invalid_validity), 0.0, invalid_validity == 0))

# per-rule counts so a failure says which rule broke
for rule_name, rule_cond in [("event_type", bad_event_type), ("lnp_type", bad_lnp_type),
                             ("sv_type", bad_sv_type), ("lrn", bad_lrn), ("spid_length", bad_spid_len)]:
    print("validity " + rule_name + " violations:", silver.filter(rule_cond).count())

# currency: ingestion_ts within max age.
CURRENCY_MAX_AGE_DAYS = 7
stale = silver.filter(F.to_date("ingestion_ts") < F.date_sub(F.current_date(), CURRENCY_MAX_AGE_DAYS)).count()
rows.append(("currency", "ingestion_ts_within_max_age", float(stale), 0.0, stale == 0))

validity event_type violations: 0
validity lnp_type violations: 0
validity sv_type violations: 0
validity lrn violations: 0
validity spid_length violations: 0

Connection: project.spark.compatibility | Run start time: 2026-07-30 20:49:52.794442 | Run duration : 0:01:01.893923s.


In [18]:
%%pyspark project.spark.compatibility
# timeliness: no event should be later than ingestion_ts.
late = silver.filter(F.col("event_timestamp") > F.col("ingestion_ts")).count()
rows.append(("timeliness", "events_after_ingestion_ts", float(late), 0.0, late == 0))



Connection: project.spark.compatibility | Run start time: 2026-07-30 20:50:54.693028 | Run duration : 0:00:06.995706s.


In [19]:
%%pyspark project.spark.compatibility
for r in rows:
    print(r)

dq = (spark.createDataFrame(rows, ["dimension", "check", "value", "threshold", "passed"])
      .withColumn("dataset", lit("tu_portps_tulip"))
      .withColumn("run_ts", lit(RUN_TS).cast("timestamp")))
dq.show(50, False)
dq.write.mode("overwrite").parquet(DQ_RESULTS + "run_date=" + RUN_TS.strftime("%Y-%m-%d") + "/")

('completeness', 'non_null_request_id', 1.0, 1.0, True)
('completeness', 'non_null_record_id', 1.0, 1.0, True)
('completeness', 'non_null_phone_number_AC_hash', 1.0, 1.0, True)
('completeness', 'non_null_phone_number_AT_hash', 1.0, 1.0, True)
('completeness', 'non_null_event_timestamp', 1.0, 1.0, True)
('completeness', 'non_null_event_date', 1.0, 1.0, True)
('completeness', 'non_null_event_type', 1.0, 1.0, True)
('completeness', 'non_null_event_seq', 1.0, 1.0, True)
('completeness', 'non_null_lnp_type', 1.0, 0.0, True)
('completeness', 'non_null_is_inter_carrier_port', 1.0, 1.0, True)
('completeness', 'non_null_from_spid', 0.9022936811333483, 0.0, True)
('completeness', 'non_null_from_mno_raw', 0.8945356420058467, 0.0, True)
('completeness', 'non_null_from_mno_std', 0.7340903980211378, 0.0, True)
('completeness', 'non_null_to_spid', 1.0, 1.0, True)
('completeness', 'non_null_to_mno_raw', 0.9916797841241286, 0.0, True)
('completeness', 'non_null_to_mno_std', 0.7430852259950529, 0.0, Tru

In [20]:
%%pyspark project.spark.compatibility
# Check definitions -> dq/checks/ (check, threshold, severity).
# completeness defs are generated from MUST_NONNULL so the two cannot drift apart.
check_defs = [("completeness", "non_null_" + c, "ratio==1.0", "error") for c in MUST_NONNULL] + [
    ("uniqueness",   "distinct_request_id_ratio",      "ratio==1.0", "error"),
    ("uniqueness",   "distinct_record_id_ratio",       "ratio==1.0", "error"),
    ("uniqueness",   "distinct_natural_key_ratio",     "ratio==1.0", "error"),
    ("consistency",  "is_inter_carrier_port_matches_lnp_type", "violations==0", "error"),
    ("consistency",  "event_date_eq_timestamp_date",           "violations==0", "error"),
    ("consistency",  "event_seq_contiguous_per_phone",         "violations==0", "warn"),
    ("validity",     "values_in_allowed_domain",   "violations==0", "error"),
    ("timeliness",           "events_after_ingestion_ts",   "count==0", "warn"),
    ("currency",             "ingestion_ts_within_max_age", "violations==0", "warn"),
    ("referential_integrity","orphan_rate",               "monitor (<0.99; >=0.99=hash mismatch)", "info"),
]
spark.createDataFrame(check_defs, ["dimension", "check", "threshold", "severity"]) \
    .withColumn("dataset", lit("tu_portps_tulip")) \
    .write.mode("overwrite").parquet(DQ_CHECKS)



Connection: project.spark.compatibility | Run start time: 2026-07-30 20:51:08.645180 | Run duration : 0:00:02.216563s.


In [21]:
%%pyspark project.spark.compatibility
# Unknown SPIDs -> dq/issues/ (in data, not in the brand dict; ranked by frequency).
unk_to   = silver.filter(F.col("to_spid").isNotNull()   & F.col("to_mno_raw").isNull()).select(F.col("to_spid").alias("spid"))
unk_from = silver.filter(F.col("from_spid").isNotNull() & F.col("from_mno_raw").isNull()).select(F.col("from_spid").alias("spid"))
unknown_spids = unk_to.union(unk_from).groupBy("spid").count().orderBy(F.desc("count"))
print("unknown SPIDs (not in dict):")
unknown_spids.show(100, False)
unknown_spids.write.mode("overwrite").parquet(DQ_ISSUES + "severity=info/unknown_spid/")

# Unknown sv_type values -> dq/issues/ (values outside SV_TYPE_VOCAB, for review).
unknown_sv_type = (silver.filter(F.col("sv_type").isNotNull() & ~F.col("sv_type").isin(*SV_TYPE_VOCAB))
                   .groupBy("sv_type").count().orderBy(F.desc("count")))
print("unknown sv_type values (not in SV_TYPE_VOCAB):")
unknown_sv_type.show(50, False)
unknown_sv_type.write.mode("overwrite").parquet(DQ_ISSUES + "severity=info/unknown_sv_type/")

unknown SPIDs (not in dict):
+----+-----+
|spid|count|
+----+-----+
|582F|75   |
|101E|38   |
|8506|14   |
|709G|7    |
|8257|4    |
|103E|3    |
|4297|1    |
|069J|1    |
+----+-----+

unknown sv_type values (not in SV_TYPE_VOCAB):
+-------+-----+
|sv_type|count|
+-------+-----+
+-------+-----+

Connection: project.spark.compatibility | Run start time: 2026-07-30 20:51:10.865778 | Run duration : 0:00:20.618032s.


In [22]:
%%pyspark project.spark.compatibility
# Referential integrity vs account_changes_batch -> dq/results/.
acct       = spark.table(ACCT_CHANGES_TABLE).select("phone_number_AC_hash").distinct()
tu_keys    = silver.select("phone_number_AC_hash").distinct()
tu_total   = tu_keys.count()
acct_total = acct.count()
orphans    = tu_keys.join(acct, "phone_number_AC_hash", "left_anti").count()
rate       = (orphans / tu_total) if tu_total else None
print("orphan rate (TU keys not in account_changes_batch): %d / %d = %.4f" % (orphans, tu_total, rate or 0))

# >=99% orphan almost always means the hash does not line up, not real orphans.
note = "HASH_MISMATCH_SUSPECTED" if (rate is not None and rate >= 0.99) else "ok"
if note == "HASH_MISMATCH_SUSPECTED":
    print("[!] orphan rate >= 99% -- phone_number_AC_hash almost certainly does not match account_changes_batch.")



orphan rate (TU keys not in account_changes_batch): 1155 / 4946 = 0.2335
Connection: project.spark.compatibility | Run start time: 2026-07-30 20:51:31.487817 | Run duration : 0:01:07.281974s.


In [23]:
%%pyspark project.spark.compatibility
(spark.createDataFrame(
    [("referential_integrity", "orphan_rate", rate, orphans, tu_total, acct_total, note)],
    ["dimension", "check", "ratio", "orphans", "tu_keys", "acct_keys", "note"])
    .withColumn("run_ts", lit(RUN_TS).cast("timestamp"))
    .write.mode("overwrite").parquet(DQ_RESULTS + "referential_integrity_account_changes/"))




Connection: project.spark.compatibility | Run start time: 2026-07-30 20:52:38.774223 | Run duration : 0:00:02.243077s.


In [24]:
%%pyspark project.spark.compatibility
# Response funnel: rebuild the response-level buckets from bronze and reconcile counts.
parse_udf = udf(parse_tnresponse, parse_struct)

_read = lambda p: (spark.read.option("header", True).option("inferSchema", True)
                   .option("nullValue", "\\N").option("quote", '"').option("escape", '"')
                   .option("multiLine", True).csv(p).withColumnRenamed("ID", "bronze_row_id"))
bronze     = _read(BRONZE_PATH)
bronze_key = _read(BRONZE_JOIN_KEY_PATH)
bronze     = bronze.join(bronze_key.select("bronze_row_id", "acHash", "atHash").dropDuplicates(["bronze_row_id"]),
                         "bronze_row_id", "left")
dedup_cols   = [c for c in bronze.columns if c != "bronze_row_id"]
bronze_dedup = bronze.groupBy(*dedup_cols).agg(F.min("bronze_row_id").alias("bronze_row_id"))

ok_resp  = bronze_dedup.filter(F.col("API_ResponseCode") == OK_CODE)
non_3000 = bronze_dedup.filter(F.col("API_ResponseCode").isNull() | (F.col("API_ResponseCode") != OK_CODE))
parsed_resp = ok_resp.withColumn("p", parse_udf(F.col("Raw_XML_Response")))
bad_xml    = parsed_resp.filter(~F.col("p.parse_ok"))
good       = parsed_resp.filter(F.col("p.parse_ok"))
no_events  = good.filter(F.col("p.events").isNull() | (F.size("p.events") == 0))
has_events = good.filter(F.size("p.events") > 0)

# Split has_events into port vs non-port at the response level.
_RAW_PORT = ["Inter-SPID Port", "Intra-SPID Port"]
_hp = has_events.withColumn("_has_port", F.exists(F.col("p.events"), lambda e: e["Event"].isin(*_RAW_PORT)))
has_inter_intra = _hp.filter(F.col("_has_port"))
no_inter_intra  = _hp.filter(~F.col("_has_port"))

total_resp   = bronze_dedup.count()
n_non_3000   = non_3000.count()
n_ok_3000    = ok_resp.count()
n_bad_xml    = bad_xml.count()
n_no_events  = no_events.count()
n_has_events = has_events.count()
n_has_inter_intra = has_inter_intra.count()
n_no_inter_intra  = no_inter_intra.count()
n_events     = total

acct1 = (total_resp == n_non_3000 + n_ok_3000)                  # total = non_3000 + ok_3000
acct2 = (n_ok_3000  == n_bad_xml + n_no_events + n_has_events)  # ok_3000 = bad_xml + no_events + has_events
acct3 = (n_has_events == n_has_inter_intra + n_no_inter_intra)  # has_events = has_inter_intra + no_inter_intra

recon = (spark.createDataFrame([
        ("total_responses",      total_resp),
        ("non_3000",             n_non_3000),
        ("ok_3000",              n_ok_3000),
        ("bad_xml",              n_bad_xml),
        ("no_events",            n_no_events),
        ("has_events_responses", n_has_events),
        ("has_inter_intra_spid",  n_has_inter_intra),
        ("no_inter_intra_spid",   n_no_inter_intra),
        ("events_emitted",       n_events),
    ], ["stage", "count"])
    .withColumn("dataset", lit("tu_portps_tulip"))
    .withColumn("run_ts",  lit(RUN_TS).cast("timestamp")))
recon.show(20, False)
print("closure total == non_3000 + ok_3000        :", acct1)
print("closure ok_3000 == bad_xml+no_events+has_events :", acct2)
print("closure has_events == has+no_inter_intra    :", acct3)
recon.write.mode("overwrite").parquet(DQ_RESULTS + "response_breakdown/")

# Classify no_events responses by TU's message.
comp = F.col("p.snapshot")["Ownership/Company"]
reason = (no_events
    .withColumn("reason_class",
        F.when(comp.startswith("No customer agreement"), "us_no_agreement")
         .when(comp.startswith("NPA not found"),          "invalid_npa")
         .when(comp.contains("not been opened"),          "npanxx_not_opened")
         .when(comp.contains("does not exist"),           "npanxx_not_assigned")
         .otherwise("other"))
    .groupBy("reason_class").count().orderBy(F.desc("count")))
reason.show(20, False)
reason.write.mode("overwrite").parquet(DQ_RESULTS + "invalid_input_breakdown/")
print("done.")

+--------------------+-----+------------------+-------------------+
|stage               |count|dataset           |run_ts             |
+--------------------+-----+------------------+-------------------+
|total_responses     |9034 |tu_portps_tulip|2026-07-30 16:47:01|
|non_3000            |0    |tu_portps_tulip|2026-07-30 16:47:01|
|ok_3000             |9034 |tu_portps_tulip|2026-07-30 16:47:01|
|bad_xml             |0    |tu_portps_tulip|2026-07-30 16:47:01|
|no_events           |215  |tu_portps_tulip|2026-07-30 16:47:01|
|has_events_responses|8819 |tu_portps_tulip|2026-07-30 16:47:01|
|has_inter_intra_spid|4946 |tu_portps_tulip|2026-07-30 16:47:01|
|no_inter_intra_spid |3873 |tu_portps_tulip|2026-07-30 16:47:01|
|events_emitted      |8894 |tu_portps_tulip|2026-07-30 16:47:01|
+--------------------+-----+------------------+-------------------+

closure total == non_3000 + ok_3000        : True
closure ok_3000 == bad_xml+no_events+has_events : True
closure has_events == has+no_inter_in

In [25]:
%%pyspark project.spark.compatibility



Connection: project.spark.compatibility | Run start time: 2026-07-30 20:53:16.804910 | Run duration : 0:00:00.042396s.
